# Estimator Sheet 
This notebook builds the single combined dataset used for quantile regression estimation. It walks a folder of forecast-card CSVs (`project-*.csv`, `poi-*.csv`, `observations-*.csv`), finds every project, combines each project's files into one merged table, adds K/D factors, county-level socioeconomic data, peak-hour volume, and post-construction capacity, then writes everything out to a single `combined_output.csv`. That combined file is the input for the model-fitting notebook.

**How to use this notebook:**
1. Run the "find project folders" cell to confirm every project was located. Set `DATA_ROOT` to the folder containing all your project CSVs.
2. Run the "build the combined dataset" cell. For each project this loads its project, poi, and observations CSVs, merges them together, and labels each observation as `before` or `after` the project's actual open date. All projects are then stacked into one `master_df`.
3. Run the "years from open" cell to add a `years_from_open` column to `master_df`, the difference between each observation's year and the project's actual open year.
4. Run the "K/D factors" cell to add `k_factor` and `d_factor` to `master_df`.
5. Run the "add county-level socioeconomic data" cell to merge in population, employment, pay, and fuel-price columns from `us_county_socioeconomic_2000_2025.csv` (matched on state, county, and year). These columns land right after `d_factor` and are also what the capacity equation's population/income growth terms pull from.
6. Run the "peak-hour volume" cell to add `peak_hour_volume`. It runs after the socioeconomic merge so that, in `combined_output.csv`, the socioeconomic columns end up sitting between `d_factor` and `peak_hour_volume`.
7. Run the "post-construction capacity" cells to add `engineering_capacity_ceng`, `capacity_multiplier`, and `peak_hour_capacity` to `master_df`, via the **Post-Construction Capacity Expansion Equation**.
8. Run the "spot-check" cell to visually confirm the merges and factor calculations worked as expected.
9. Run the "write out" cell to add a blank `comment` column and save `master_df` to `combined_output.csv`.

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Find your project folders

Point `DATA_ROOT` at the folder holding your project data. This walks the entire directory tree underneath it and collects every sub-folder that contains a `project-*.csv` file (each matching folder is treated as one project). The result is a sorted list of folder paths, printed below so you can confirm the right projects were found.

**Update this path to point at your own data folder before continuing.**

In [2]:
def find_project_folders(root_dir):
    """Return every sub-folder of root_dir that contains a project-*.csv file."""
    folders = []
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        if any(re.match(r"project-.*\.csv$", f, re.IGNORECASE) for f in filenames):
            folders.append(dirpath)
    return sorted(folders)

DATA_ROOT = r"C:\Users\User\Documents\Code\forecastcards_ba_data\data"
project_folders = find_project_folders(DATA_ROOT)
project_folders

['C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237506-2',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237565-1',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237576-1',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237592-1',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237592_2',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237650-3',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_237965-1',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238394-3',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238395-4',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238395-5',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238429-4',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\FL_238429-5',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba

## 2. Build the combined dataset

This section does the actual finding, loading, and merging for every project, then stacks the results into one master table used for estimation.

**Loading a project's files** — for a given project folder, finds and reads:
- `project-*.csv` → project-level metadata (one row per project)
- `poi-*.csv` → "points of interest" tied to that project (e.g. intersections/segments)
- `observations-*.csv` → observed traffic/volume records

**Merging a project's files** — performs two sequential merges:
- **project → poi:** tags each POI row with its `project_id`, then merges project metadata onto every POI row (keeping all POIs, broadcasting the single project row across them).
- **before/after labeling:** converts `observation_date` to a real datetime and compares it to the project's `date_open_actual` to create `before_or_after`.
- **(project+poi) → observations:** merges the project/poi table onto the observations, matching on `poi_id` **and** `before_or_after`, keeping every observation row, and renames `start_time`/`end_time` to the `_observation` versions for consistent naming.


In [3]:
def read_csv_any_encoding(path, **kwargs):
    """Try UTF-8 first; fall back to Windows-1252 (Excel's default on Windows)."""
    for enc in ("utf-8", "utf-8-sig", "cp1252"):
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            if enc == "cp1252":
                print(f"Note: read {os.path.basename(path)} as cp1252")
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"Could not decode {path}")
    # non-breaking spaces in headers would break lookups like df["project_id"]
    df.columns = df.columns.str.replace("\xa0", " ").str.strip()
    # drop blank header-less columns left by trailing commas (e.g. "Unnamed: 8")
    unnamed = [c for c in df.columns if c.startswith("Unnamed:") and df[c].isna().all()]
    df = df.drop(columns=unnamed)
    return df


# Some projects use different names for the same field. Map them onto one name
# so they land in a single column in combined_output.csv.
COLUMN_ALIASES = {
    "milepoint": "segment_length_miles",
    "speed_limit_mph": "speed_limit",
}

def apply_column_aliases(df):
    for old, new in COLUMN_ALIASES.items():
        if old not in df.columns:
            continue
        if new in df.columns:
            df[new] = df[new].fillna(df[old])
            df = df.drop(columns=old)
        else:
            df = df.rename(columns={old: new})
    return df


def load_project_files(folder):
    """Load the project / poi / observations csvs for one folder."""
    project_path = glob.glob(os.path.join(folder, "project-*.csv"))[0]
    poi_path = glob.glob(os.path.join(folder, "poi-*.csv"))[0]
    obs_path = glob.glob(os.path.join(folder, "observations-*.csv"))[0]

    project_df = read_csv_any_encoding(project_path).dropna(how="all")
    poi_df = read_csv_any_encoding(poi_path, dtype={"poi_id": str}).dropna(how="all")
    obs_df = read_csv_any_encoding(obs_path, dtype={"poi_id": str}).dropna(how="all")

    project_df = apply_column_aliases(project_df)
    poi_df = apply_column_aliases(poi_df)
    obs_df = apply_column_aliases(obs_df)

    return project_df, poi_df, obs_df

In [4]:
def process_project_folder(folder):
    project_df, poi_df, obs_df = load_project_files(folder)

    project_df = project_df.copy()

    comment_cols = [c for c in project_df.columns if c.strip().lower() in ("comment", "comments")]
    if comment_cols:
        combined_comment = project_df[comment_cols[0]]
        for extra_col in comment_cols[1:]:
            combined_comment = combined_comment.fillna(project_df[extra_col])
        project_df = project_df.drop(columns=comment_cols)
        project_df["comment"] = combined_comment
    else:
        project_df["comment"] = pd.NA

    project_id = project_df["project_id"].iloc[0]
    date_open_actual = pd.to_datetime(project_df["date_open_actual"].iloc[0], errors="coerce")
    date_open_planned = pd.to_datetime(project_df["date_open_planned"].iloc[0], errors="coerce")

    # use the actual open date when available; otherwise fall back to the planned open date
    date_open_effective = date_open_actual if pd.notna(date_open_actual) else date_open_planned

    if pd.isna(date_open_effective):
        print(f"WARNING: no usable open date for project {project_id} in {folder}")

    #match POIs ignoring leading zeros
    def poi_key(s):
        return s.astype("string").str.strip().str.lstrip("0")

    poi_df = poi_df.copy()

    if "before_or_after" in poi_df.columns:
        poi_df["before_or_after"] = (
            poi_df["before_or_after"].astype("string").str.strip().str.lower()
        )
    poi_df["project_id"] = project_id
    poi_df["_poi_key"] = poi_key(poi_df["poi_id"])                       

    # STEP 1: project -> poi
    project_poi = pd.merge(project_df, poi_df, on="project_id", how="right")

    # derive a sensible before/after label for each observation
    obs_df = obs_df.copy()
    obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"], errors="coerce")
    obs_df = obs_df.dropna(subset=["observation_date"])
    obs_df["before_or_after"] = pd.array(
        np.where(obs_df["observation_date"] < date_open_effective, "before", "after"),
        dtype="string",
    )
    obs_df["before_or_after_binary"] = (obs_df["before_or_after"] == "after").astype(int)
    obs_df["_poi_key"] = poi_key(obs_df["poi_id"])                       

    # STEP 2: (project + poi) -> observation
    # keep every observation row; observations with no matching POI get blank POI columns
    merged = pd.merge(
        project_poi, obs_df,
        on=["_poi_key", "before_or_after"],
        how="right",
        suffixes=("", "_obs"),
        indicator=True,
    )
    merged["poi_id"] = merged["poi_id_obs"]

    # unmatched rows came through with blank project columns too -- refill those,
    # since the project info is the same for every row in this folder
    for col in project_df.columns:
        if col in merged.columns:
            merged[col] = project_df[col].iloc[0]

    unmatched = merged[merged["_merge"] == "right_only"]
    if len(unmatched) > 0:
        print(f"Project {project_id}: kept {len(unmatched)} observation rows with no matching POI "
              f"(POI columns left blank): {sorted(unmatched['poi_id'].dropna().unique())}")

    merged = merged.drop(columns=["_poi_key", "poi_id_obs", "_merge"])

    merged = merged.rename(columns={"start_time": "start_time_observation",
                                     "end_time": "end_time_observation"})

    merged["date_open_effective"] = date_open_effective

    return merged

In [5]:
project_tables = [process_project_folder(folder) for folder in project_folders]

master_df = pd.concat(project_tables, ignore_index=True, sort=False)

print(f"{len(master_df)} total observation rows across {len(project_tables)} project(s)")
master_df.head(10)

Note: read poi-238394-3.csv as cp1252
Project 238429-4: kept 14 observation rows with no matching POI (POI columns left blank): ['111012']
Project 238753-1A: kept 28 observation rows with no matching POI (POI columns left blank): ['360024', '360030']
Project 240805-2: kept 19 observation rows with no matching POI (POI columns left blank): ['791011']
Project 6-313.50: kept 9 observation rows with no matching POI (POI columns left blank): ['059G10']
Project 6-356.00: kept 9 observation rows with no matching POI (POI columns left blank): ['094A04']
Project 7-113.01: kept 46 observation rows with no matching POI (POI columns left blank): ['034E49', '034E54', '034E58']
Project 7-8405.00: kept 10 observation rows with no matching POI (POI columns left blank): ['009A80']
Project 20200057: kept 9 observation rows with no matching POI (POI columns left blank): ['02489002', '02489004', '02489005', '02489006', '02489007', '02489008', '02489009', '02489010', '02489011']
7284 total observation rows

,project_id,state,county,project_type,description,date_open_planned,date_horizon,construction_start_date,date_open_actual,area_type,project_length_miles,comment,poi_id,before_or_after,facility_name,segment_length_miles,speed_limit,lane_width,signals_per_mile,stop_signs_per_mile,through_lanes,left_turn_lanes,right_turn_lanes,express_hov_hot_lanes,shoulder_width,functional_class,obs_id,observation_date,start_time_observation,end_time_observation,observation_variable,observation_value,observation_unit,before_or_after_binary,date_open_effective
0,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_1,1997-01-01,0:00:00,24:00:00,volume,29800,AADT,0,2007-01-01
1,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_2,1998-01-01,0:00:00,24:00:00,volume,30500,AADT,0,2007-01-01
2,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_3,1999-01-01,0:00:00,24:00:00,volume,30100,AADT,0,2007-01-01
3,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_4,2000-01-01,0:00:00,24:00:00,volume,30500,AADT,0,2007-01-01
4,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,before,S.R. 520,13.108,55,10,0,0,4,0,0,0,6,principal arterial,701007_5,2001-01-01,0:00:00,24:00:00,volume,30500,AADT,0,2007-01-01
5,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_6,2010-01-01,0:00:00,24:00:00,volume,27000,AADT,1,2007-01-01
6,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_7,2011-01-01,0:00:00,24:00:00,volume,27000,AADT,1,2007-01-01
7,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_8,2012-01-01,0:00:00,24:00:00,volume,22500,AADT,1,2007-01-01
8,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_9,2013-01-01,0:00:00,24:00:00,volume,26000,AADT,1,2007-01-01
9,237506-2,FL,Brevard,capacity expansion,"Banana River along S.R. 520, beginning at Mil...",1/1/2003,1/1/2023,unknown,1/1/2007,urban,0.6,<NA>,701007,after,S.R. 520,13.108,55,12,0,0,4,0,0,0,13,principal arterial,701007_10,2014-01-01,0:00:00,24:00:00,volume,25500,AADT,1,2007-01-01


## 3. Years from open (before/after completion year)

Adds a `years_from_open` column to `master_df`: the difference between the calendar year of each observation and the calendar year the project actually opened (`date_open_actual`).

In [6]:
master_df["observation_date"] = pd.to_datetime(master_df["observation_date"], errors="coerce")
master_df["date_open_actual"] = pd.to_datetime(master_df["date_open_actual"], errors="coerce")
master_df["date_open_planned"] = pd.to_datetime(master_df["date_open_planned"], errors="coerce")

# use the actual open date when available; otherwise fall back to the planned open date
master_df["date_open_effective"] = master_df["date_open_actual"].fillna(master_df["date_open_planned"])

master_df["years_from_open"] = (
    master_df["observation_date"].dt.year - master_df["date_open_effective"].dt.year
)

master_df[["project_id", "poi_id", "observation_date", "date_open_actual", "date_open_planned", "date_open_effective", "years_from_open"]].head(10)

,project_id,poi_id,observation_date,date_open_actual,date_open_planned,date_open_effective,years_from_open
0,237506-2,701007,1997-01-01,2007-01-01,2003-01-01,2007-01-01,-10.0
1,237506-2,701007,1998-01-01,2007-01-01,2003-01-01,2007-01-01,-9.0
2,237506-2,701007,1999-01-01,2007-01-01,2003-01-01,2007-01-01,-8.0
3,237506-2,701007,2000-01-01,2007-01-01,2003-01-01,2007-01-01,-7.0
4,237506-2,701007,2001-01-01,2007-01-01,2003-01-01,2007-01-01,-6.0
5,237506-2,701007,2010-01-01,2007-01-01,2003-01-01,2007-01-01,3.0
6,237506-2,701007,2011-01-01,2007-01-01,2003-01-01,2007-01-01,4.0
7,237506-2,701007,2012-01-01,2007-01-01,2003-01-01,2007-01-01,5.0
8,237506-2,701007,2013-01-01,2007-01-01,2003-01-01,2007-01-01,6.0
9,237506-2,701007,2014-01-01,2007-01-01,2003-01-01,2007-01-01,7.0


## 4. K/D factors

Adds `k_factor` and `d_factor` using `K_D_Factors.csv`, matched to each row's `functional_class`:

- `k_factor`, `d_factor` — looked up directly from `K_D_Factors.csv`.

Rows with `functional_class == "unknown"` will come out blank for these columns.

In [7]:
# Paths to the factor tables
FACTOR_TABLES_ROOT = r"C:\Users\User\Documents\Code\forecastcards_v2\forecastcards\capacity_factor_tables"

K_D_FACTORS_PATH = os.path.join(FACTOR_TABLES_ROOT, "K_D_Factors.csv")
CAP_CAL_PATH = os.path.join(FACTOR_TABLES_ROOT, "Cap_Cal_Values.csv")
CAP_BASE_PATH = os.path.join(FACTOR_TABLES_ROOT, "Cap_Base_Values.csv")

# The canonical functional-class buckets shared by K_D_Factors.csv and Cap_Base_Values.csv.
CANONICAL_FUNCTIONAL_CLASSES = {
    "freeway", "principal arterial", "minor arterial",
    "major collector", "minor collector", "local", "ramp",
}

def normalize_functional_class(value):
    """Normalize a functional_class label onto the canonical set of classes shared by
    K_D_Factors.csv and Cap_Base_Values.csv (freeway / principal arterial / minor arterial /
    major collector / minor collector / local / ramp). Handles:
      - case/whitespace differences
      - Cap_Base_Values.csv's "Local Road" vs K_D_Factors.csv's "Local"
      - KY's "<Class> - <Qualifier>" suffixes, e.g. "Principal Arterial - Other"
      - "Interstate" as a synonym for "Freeway"
      - a known data-entry typo, "Princicpal Arterial"
    Anything that still doesn't match a canonical class after this (e.g. "unknown") is
    returned as-is, and will simply come out blank in the k_factor/d_factor/Cap_Base lookups.
    """
    if pd.isna(value):
        return None
    label = str(value).strip().lower()

    if "freeways and expressway" in label:
        return "freeway"

    if " - " in label:
        label = label.split(" - ")[0].strip()

    # known data-entry typo ("Princicpal Arterial")
    label = label.replace("princicpal", "principal")

    # synonyms / alternate spellings for the same canonical class
    SYNONYMS = {
        "interstate": "freeway",
        "local road": "local",
        # FL / NE labels that were falling through to blank K/D (see the WARNING this cell printed)
        "other principal arterial": "principal arterial",
        "primary arterial": "principal arterial",
        "principal arterials": "principal arterial",
        # bare "arterial" (FL) doesn't say principal vs minor; treated as principal arterial.
        # Change to "minor arterial" if that's what these segments actually are.
        "arterial": "principal arterial",
    }
    label = SYNONYMS.get(label, label)

    return label

# K / D factors by functional class
kd_factors = read_csv_any_encoding(K_D_FACTORS_PATH)[["Functional Class", "K Factor", "D Factor"]].dropna(subset=["Functional Class"])
kd_factors["_fc"] = kd_factors["Functional Class"].apply(normalize_functional_class)
k_factor_lookup = kd_factors.set_index("_fc")["K Factor"].to_dict()
d_factor_lookup = kd_factors.set_index("_fc")["D Factor"].to_dict()

cap_base = read_csv_any_encoding(CAP_BASE_PATH)
cap_base["_fc"] = cap_base["Facility Type"].apply(normalize_functional_class)
cap_base_lookup = cap_base.set_index("_fc")["Base Capacity (veh/hr/lane)"].to_dict()

# Sanity check: K_D_Factors.csv and Cap_Base_Values.csv should normalize to the exact same
# set of canonical classes. If they don't, the two tables have drifted apart.
kd_classes = set(k_factor_lookup.keys())
cb_classes = set(cap_base_lookup.keys())
if kd_classes != CANONICAL_FUNCTIONAL_CLASSES or cb_classes != CANONICAL_FUNCTIONAL_CLASSES:
    print("WARNING: K_D_Factors.csv / Cap_Base_Values.csv classes don't match the expected "
          f"canonical set. K_D_Factors: {sorted(kd_classes)} | Cap_Base_Values: {sorted(cb_classes)} "
          f"| expected: {sorted(CANONICAL_FUNCTIONAL_CLASSES)}")

master_df["through_lanes"] = pd.to_numeric(master_df["through_lanes"], errors="coerce")

_fc_norm = master_df["functional_class"].apply(normalize_functional_class)
master_df["k_factor"] = _fc_norm.map(k_factor_lookup)
master_df["d_factor"] = _fc_norm.map(d_factor_lookup)

_unmatched = sorted(
    v for v in _fc_norm.dropna().unique()
    if v not in CANONICAL_FUNCTIONAL_CLASSES and v != "unknown"
)
if _unmatched:
    print(f"WARNING: functional_class values not recognized after normalization: {_unmatched} "
          "-- these rows will have blank k_factor/d_factor. Add them to normalize_functional_class.")

# --- Align state-specific area_type / project_type labeling conventions ---

KY_AREA_TYPE_MAP = {
    "large urbanized area": "urban",
    "small urbanized area": "urban",
    "small urban area": "suburban",
    "rural area": "rural",
}

def normalize_area_type(value):
    if pd.isna(value):
        return None
    label = str(value).strip().lower()
    return KY_AREA_TYPE_MAP.get(label, label)

master_df["area_type_normalized"] = master_df["area_type"].apply(normalize_area_type)

PROJECT_TYPE_CATEGORY_MAP = {
    # MN
    "road-capacity-expansion": "widening",
    "new-roadway": "new construction",
    # KY
    "major widen": "widening",
    "ichg reconst": "widening",
    "new road": "new construction",
    "new route": "new construction",
    "new interchange": "new construction",
}

def normalize_project_type(value):
    if pd.isna(value):
        return None
    label = str(value).strip().lower()
    return PROJECT_TYPE_CATEGORY_MAP.get(label, label)

master_df["project_type_category"] = master_df["project_type"].apply(normalize_project_type)

master_df[["functional_class", "through_lanes", "k_factor", "d_factor",
           "area_type", "area_type_normalized", "project_type", "project_type_category"]].head(10)

,functional_class,through_lanes,k_factor,d_factor,area_type,area_type_normalized,project_type,project_type_category
0,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
1,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
2,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
3,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
4,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
5,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
6,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
7,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
8,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion
9,principal arterial,4.0,0.12,0.6,urban,urban,capacity expansion,capacity expansion


## 5. Add county-level socioeconomic data

Merges in columns from `us_county_socioeconomic_2000_2025.csv` (population, employment, avg annual pay, fuel prices, and their 2025-dollar equivalents) onto `master_df`, matched on **state + county + year**:

- `state` (abbreviation, e.g. `MN`) is mapped to the socioeconomic file's full state name.
- `county` is normalized by stripping the 'County' suffixes so `"Ramsey"` matches `"Ramsey County"`.
- Combined-county POIs (e.g. `"Hennepin and Ramsey"`) are matched by averaging the socioeconomic values across each constituent county for that year.
- The year used for matching is the calendar year of `observation_date`.
- The socioeconomic file's own `State`, `County`, and `Year` columns are **not** included in the output.
- Rows with no match (e.g. observation years outside 2000–2025) come out blank for these columns.

This runs right after `k_factor`/`d_factor` are added and before `peak_hour_volume`, so these columns land between `d_factor` and `peak_hour_volume` in the final `combined_output.csv`. It also has to run before the Post-Construction Capacity Expansion Equation below, since that equation's `Δln(Pop_r)` and `Δln(Inc_r)` terms are built from the `population` and `avg annual pay (2025$)` columns added here.

In [8]:
US_STATE_ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO",
    "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    "District of Columbia": "DC",
}

COUNTY_SUFFIXES = [
    " County", " Parish", " Borough", " Census Area", " Municipality",
    " Municipio", " City and Borough", " city", " City", " Region",
]

def normalize_county(name):
    # Strip common county/parish/borough suffixes so names match across files.
    if pd.isna(name):
        return None
    name = str(name).strip()
    for suf in COUNTY_SUFFIXES:
        if name.endswith(suf):
            return name[: -len(suf)].strip()
    return name

SOCIO_PATH = os.path.join(FACTOR_TABLES_ROOT, "us_county_socioeconomic_2000_2025.csv")

socio_df = read_csv_any_encoding(SOCIO_PATH)
socio_df["state_abbrev"] = socio_df["State"].map(US_STATE_ABBREV)
socio_df["county_clean"] = socio_df["County"].apply(normalize_county)

SOCIO_VALUE_COLS = [
    c for c in socio_df.columns
    if c not in ("State", "County", "Year", "state_abbrev", "county_clean")
]

socio_lookup = socio_df.groupby(["state_abbrev", "county_clean", "Year"])[SOCIO_VALUE_COLS].mean()

master_df["observation_year"] = pd.to_datetime(master_df["observation_date"], errors="coerce").dt.year

master_df["county"] = master_df["county"].apply(normalize_county)
master_df["state"] = master_df["state"].astype(str).str.strip().str.upper()

def lookup_socio(row):
    key = (row["state"], row["county"], row["observation_year"])
    if key in socio_lookup.index:
        return socio_lookup.loc[key]
    return pd.Series({c: np.nan for c in SOCIO_VALUE_COLS})

socio_matched = master_df.apply(lookup_socio, axis=1)
socio_matched.columns = [c.lower() for c in socio_matched.columns]
n_unmatched = socio_matched.isna().all(axis=1).sum()
if n_unmatched:
    print(f"WARNING: {n_unmatched} row(s) had no socioeconomic match (check state/county/year):")
    print(master_df.loc[socio_matched.isna().all(axis=1), ["state", "county", "observation_year"]].drop_duplicates())

master_df = pd.concat([master_df, socio_matched], axis=1).drop(columns=["observation_year"])

master_df[["state", "county", "d_factor"] + list(socio_matched.columns)].head(10)

     state   county  observation_year
0       FL  Brevard              1997
1       FL  Brevard              1998
2       FL  Brevard              1999
101     FL  Brevard              1996
366     FL  Flagler              1999
...    ...      ...               ...
4857    KY    Rowan              1998
4866    KY    Rowan              1996
4867    KY    Rowan              1999
4897    KY    Mason              1998
6172    MN   Ramsey              1997

[123 rows x 3 columns]


,state,county,d_factor,population,employment,avg annual pay,gasoline price,diesel price,cpi adjustment factor (to 2025$),avg annual pay (2025$),gasoline price (2025$),diesel price (2025$)
0,FL,Brevard,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FL,Brevard,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FL,Brevard,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FL,Brevard,0.6,477819.0,180459.0,32080.0,1.5285,1.374643,1.869588,59976.37,2.86,2.57
4,FL,Brevard,0.6,486429.0,184725.0,32798.0,1.4720,1.279747,1.817860,59622.17,2.68,2.33
5,FL,Brevard,0.6,543990.0,188093.0,43770.0,2.7450,2.670000,1.476423,64623.06,4.05,3.94
6,FL,Brevard,0.6,544393.0,187114.0,44185.0,3.4775,3.417500,1.431246,63239.60,4.98,4.89
7,FL,Brevard,0.6,547029.0,187556.0,44153.0,3.6500,3.552500,1.402227,61912.55,5.12,4.98
8,FL,Brevard,0.6,550349.0,187497.0,44364.0,3.4950,3.535000,1.381985,61310.37,4.83,4.89
9,FL,Brevard,0.6,555669.0,189748.0,44484.0,3.5075,3.485000,1.359924,60494.87,4.77,4.74


## 6. Peak-hour volume

Adds `peak_hour_volume` = `AADT x K x D`, computed from the observed volume column (only where the unit is `AADT`), using the `k_factor` / `d_factor` from Section 4. This runs after the socioeconomic merge (Section 5) so that, column-wise, the socioeconomic fields sit between `d_factor` and `peak_hour_volume` in `combined_output.csv`.

In [9]:
master_df["peak_hour_volume"] = np.where(
    master_df["observation_unit"] == "AADT",
    master_df["observation_value"] * master_df["k_factor"] * master_df["d_factor"],
    np.nan,
)

master_df[[
    "functional_class", "through_lanes", "k_factor", "d_factor",
    "population", "avg annual pay (2025$)",
    "observation_value", "peak_hour_volume",
]].head(10)

,functional_class,through_lanes,k_factor,d_factor,population,avg annual pay (2025$),observation_value,peak_hour_volume
0,principal arterial,4.0,0.12,0.6,NaN,NaN,29800,2145.6
1,principal arterial,4.0,0.12,0.6,NaN,NaN,30500,2196.0
2,principal arterial,4.0,0.12,0.6,NaN,NaN,30100,2167.2
3,principal arterial,4.0,0.12,0.6,477819.0,59976.37,30500,2196.0
4,principal arterial,4.0,0.12,0.6,486429.0,59622.17,30500,2196.0
5,principal arterial,4.0,0.12,0.6,543990.0,64623.06,27000,1944.0
6,principal arterial,4.0,0.12,0.6,544393.0,63239.60,27000,1944.0
7,principal arterial,4.0,0.12,0.6,547029.0,61912.55,22500,1620.0
8,principal arterial,4.0,0.12,0.6,550349.0,61310.37,26000,1872.0
9,principal arterial,4.0,0.12,0.6,555669.0,60494.87,25500,1836.0


## 7. Post-Construction Capacity Expansion Equation

Replaces the old flat `Cap_base x N x f_hv x PHF x f_p x f_g` lookup with the full model from
*Proposed Post-Capacity Expansion Equation (rev2)*:

$$\text{PostCapacity}_i = C_{eng,i} \times \exp\big(\beta_{LM}\Delta LM_i + \beta_L \Delta L_i + \beta_{LT}\Delta LT_i + \beta_{RT}\Delta RT_i + \beta_W W_i + \beta_E E_i + \beta_S S_i + \beta_F F_i + \beta_A \Delta\ln(AADT_i) + \beta_P \Delta\ln(Pop_r) + \beta_Y \Delta\ln(Inc_r) + \beta_{SL} SL_i + \beta_{SIG} SIG_i + \beta_{AT} AT_i\big)$$

- Before rows: $C_{eng,pre} \times e^{\beta_{SIG}SIG_{pre} + \beta_{AT}AT}$. 
- After rows: $C_{eng,post} \times e^{\text{change terms} + \beta_{SIG}SIG_{post} + \beta_{AT}AT}$.

**$C_{eng}$ (engineering baseline)** is computed with the SPR20-597 HERS free-flow-speed method (Section 3
of the doc): FFS → lane-width penalty → signal adjustment → stop-sign adjustment → speed at capacity
(90% of FFS, per HCM density = 35 pc/mi/ln) → flow per lane → x lanes x heavy-vehicle factor. It uses each
POI's **post-construction** geometry, per the equation's definition of $C_{eng}$.

In [10]:
# Load calibration betas (and HCM scalars) from Cap_Cal_Values.csv.

cap_cal = pd.read_csv(CAP_CAL_PATH, encoding="cp1252")

PARAM_TO_KEY = {
    "Heavy Vehicle Factor": "f_hv",
    "Peak Hour Factor": "PHF",
    "Driver Population Factor": "f_p",
    "Grade Adjustment Factor": "f_g",
    "Lanemiles Elasticity": "BLM",
    "Lane Count Effect": "BL",
    "Left Turn Lane Effect": "BLT",
    "Right Turn Lane Effect": "BRT",
    "Lane Width Effect": "BW",
    "Managed Lane Effect": "BE",
    "Shoulder Widening Effect": "BS",
    "Functional Class Effect": "BF",
    "Speed Limit Effect": "BSL",
    "Signals per Mile Effect": "BSIG",
    "Area Type Effect": "BAT",
    "Local Demand Change": "BA",
    "Population Growth": "BP",
    "Income Growth": "BY",
}
cap_cal["_key"] = cap_cal["Parameter"].str.strip().map(PARAM_TO_KEY)
missing = cap_cal[cap_cal["_key"].isna()]
if len(missing):
    print("WARNING: unrecognized rows in Cap_Cal_Values.csv (not used):", missing["Parameter"].tolist())

cap_cal["_value"] = pd.to_numeric(cap_cal["Default Value"], errors="coerce")

bad_parse = cap_cal[cap_cal["_value"].isna()]
if len(bad_parse):
    print("WARNING: could not parse Default Value for:", bad_parse["Parameter"].tolist())

betas = cap_cal.dropna(subset=["_key"]).set_index("_key")["_value"].to_dict()

# Sanity check against the doc's Section 5 priors
for sym in ["BSIG", "BAT"]:
    if sym in betas and betas[sym] > 0:
        print(f"WARNING: {sym} = {betas[sym]} parsed as positive but the doc expects it negative "
              f"(see Proposed_Post-Capacity_Expansion_Equation_rev2, Section 5) - check Cap_Cal_Values.csv.")

f_hv = betas["f_hv"]  # heavy-vehicle factor, used in the Ceng calc below

betas

{'f_hv': 0.95,
 'PHF': 0.92,
 'f_p': 0.95,
 'f_g': 1.0,
 'BLM': 0.45,
 'BL': 0.35,
 'BLT': 0.1,
 'BRT': 0.06,
 'BW': 0.05,
 'BE': 0.15,
 'BS': 0.05,
 'BF': 0.1,
 'BSL': 0.05,
 'BSIG': -0.1,
 'BAT': -0.1,
 'BA': 1.0,
 'BP': 0.25,
 'BY': 0.25}

In [11]:
# Per-POI before/after geometry, used both for Ceng (post-construction geometry)
# and for the Delta/indicator variables in the exponential multiplier.
GEOM_COLS = [
    "through_lanes", "lane_width", "left_turn_lanes", "right_turn_lanes",
    "shoulder_width", "express_hov_hot_lanes", "speed_limit",
    "signals_per_mile", "stop_signs_per_mile", "segment_length_miles",
]
for c in GEOM_COLS:
    master_df[c] = pd.to_numeric(master_df[c], errors="coerce")

geom_by_period = (
    master_df.groupby(["poi_id", "before_or_after"])[GEOM_COLS].mean().unstack("before_or_after")
)

def pre(col):
    return geom_by_period[col]["before"] if "before" in geom_by_period[col] else np.nan

def post(col):
    return geom_by_period[col]["after"] if "after" in geom_by_period[col] else np.nan

# post-construction functional class / area type, used for Ceng flow-type and AT level
# (area_type_normalized, from Section 4, so KY's FHWA area-type labels line up with MN's)
_after_rows = master_df[master_df["before_or_after"] == "after"]
fc_post = _after_rows.groupby("poi_id")["functional_class"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)
at_post = _after_rows.groupby("poi_id")["area_type_normalized"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)
_before_rows = master_df[master_df["before_or_after"] == "before"]
fc_pre = _before_rows.groupby("poi_id")["functional_class"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)

poi = pd.DataFrame(index=geom_by_period.index)

# --- geometry deltas (post - pre / pre) ---
# d_lane_miles is multiplied by BLM, the lane-miles ELASTICITY, so it is the log change
# ln(post / pre), not the percent change (post - pre) / pre.
lane_miles_pre = pre("through_lanes") * pre("segment_length_miles")
lane_miles_post = post("through_lanes") * post("segment_length_miles")
d_lane_miles_full = np.log(lane_miles_post / lane_miles_pre)
d_lane_miles_lanes_only = np.log(post("through_lanes") / pre("through_lanes"))
poi["d_lane_miles"] = d_lane_miles_full.where(d_lane_miles_full.notna(), d_lane_miles_lanes_only)
poi["d_through_lanes"] = post("through_lanes") - pre("through_lanes")
poi["d_left_turn_lanes"] = post("left_turn_lanes") - pre("left_turn_lanes")
poi["d_right_turn_lanes"] = post("right_turn_lanes") - pre("right_turn_lanes")

# --- 0/1 indicators (W, E, S, SL) ---
poi["lane_width_improved"] = (post("lane_width") > pre("lane_width")).astype("Int64")
poi["express_lane_added"] = (post("express_hov_hot_lanes") > pre("express_hov_hot_lanes")).astype("Int64")
poi["shoulder_widened"] = (post("shoulder_width") > pre("shoulder_width")).astype("Int64")
poi["speed_limit_increased"] = (post("speed_limit") > pre("speed_limit")).astype("Int64")

# --- functional class improvement (F): no ranking data, so left at 0 unless enriched later ---
poi["functional_class_improved"] = 0

# --- levels used directly (SIG, AT), from post-construction state ---
poi["signals_per_mile_post"] = post("signals_per_mile").fillna(0)
poi["stop_signs_per_mile_post"] = post("stop_signs_per_mile").fillna(0)
poi["through_lanes_post"] = post("through_lanes")
poi["lane_width_post"] = post("lane_width")
poi["speed_limit_post"] = post("speed_limit")
poi["functional_class_post"] = fc_post

# --- pre-construction geometry, used for the pre-construction Ceng (Section 7 base capacity).
# Any pre value that's missing falls back to the post value (i.e. assume that attribute didn't change).
poi["through_lanes_pre"] = pre("through_lanes").fillna(poi["through_lanes_post"])
poi["lane_width_pre"] = pre("lane_width").fillna(poi["lane_width_post"])
poi["speed_limit_pre"] = pre("speed_limit").fillna(poi["speed_limit_post"])
poi["signals_per_mile_pre"] = pre("signals_per_mile").fillna(poi["signals_per_mile_post"])
poi["stop_signs_per_mile_pre"] = pre("stop_signs_per_mile").fillna(poi["stop_signs_per_mile_post"])
poi["functional_class_pre"] = fc_pre.reindex(poi.index).fillna(poi["functional_class_post"])
poi["area_type_post"] = at_post
poi["area_type_level"] = at_post.astype(str).str.strip().str.lower().isin(["urban", "suburban"]).astype(int)

# --- demand delta: mean AADT pre vs post per POI ---
master_df["observation_value"] = pd.to_numeric(master_df["observation_value"], errors="coerce")
aadt_obs = master_df[master_df["observation_unit"] == "AADT"]
aadt_by_period = aadt_obs.groupby(["poi_id", "before_or_after"])["observation_value"].mean().unstack("before_or_after")
poi["d_ln_aadt"] = np.log(aadt_by_period["after"] / aadt_by_period["before"])

# --- regional population & income growth: pre vs post, straight from the county
# socioeconomic merge (Section 5) for each POI's own state/county/year ---
SOCIO_GROWTH_COLS = ["population", "avg annual pay (2025$)"]
socio_by_period = (
    master_df.groupby(["poi_id", "before_or_after"])[SOCIO_GROWTH_COLS].mean().unstack("before_or_after")
)

def socio_pre(col):
    return socio_by_period[col]["before"] if "before" in socio_by_period[col] else np.nan

def socio_post(col):
    return socio_by_period[col]["after"] if "after" in socio_by_period[col] else np.nan

poi["d_ln_pop"] = np.log(socio_post("population") / socio_pre("population"))
poi["d_ln_income"] = np.log(socio_post("avg annual pay (2025$)") / socio_pre("avg annual pay (2025$)"))

poi.head(10)

,d_lane_miles,d_through_lanes,d_left_turn_lanes,d_right_turn_lanes,lane_width_improved,express_lane_added,shoulder_widened,speed_limit_increased,functional_class_improved,signals_per_mile_post,stop_signs_per_mile_post,through_lanes_post,lane_width_post,speed_limit_post,functional_class_post,through_lanes_pre,lane_width_pre,speed_limit_pre,signals_per_mile_pre,stop_signs_per_mile_pre,functional_class_pre,area_type_post,area_type_level,d_ln_aadt,d_ln_pop,d_ln_income
poi_id,,,,,,,,,,,,,,,,,,,,,,,,,,
001047,0.0,0.0,NaN,NaN,0,0,0,0,0,0.0,0.0,2.0,12.0,55.0,Principal Arterial,2.0,12.0,55.0,0.0,0.0,Principal Arterial,rural,0.0,0.100119,0.027369,0.039450
001A69,0.0,0.0,NaN,NaN,0,0,0,0,0,0.0,0.0,2.0,12.0,55.0,Principal Arterial,2.0,12.0,55.0,0.0,0.0,Principal Arterial,rural,0.0,-0.104288,0.025049,0.036240
005017,NaN,NaN,NaN,NaN,0,0,0,0,0,0.0,0.0,2.0,11.0,55.0,unknown,2.0,11.0,55.0,0.0,0.0,unknown,rural,0.0,-0.148732,0.085283,0.010912
005021,NaN,NaN,NaN,NaN,0,0,0,0,0,0.0,0.0,2.0,11.0,55.0,unknown,2.0,11.0,55.0,0.0,0.0,unknown,rural,0.0,-0.018640,0.108920,0.038134
005602,NaN,NaN,NaN,NaN,0,0,0,0,0,0.0,0.0,6.0,12.0,70.0,unknown,6.0,12.0,70.0,0.0,0.0,unknown,rural,0.0,NaN,NaN,NaN
005603,NaN,NaN,NaN,NaN,0,0,0,0,0,0.0,0.0,6.0,12.0,70.0,unknown,6.0,12.0,70.0,0.0,0.0,unknown,rural,0.0,NaN,NaN,NaN
005604,NaN,NaN,NaN,NaN,0,0,0,0,0,0.0,0.0,6.0,12.0,70.0,unknown,6.0,12.0,70.0,0.0,0.0,unknown,rural,0.0,NaN,NaN,NaN
005606,NaN,NaN,NaN,NaN,0,0,0,0,0,0.0,0.0,6.0,12.0,70.0,unknown,6.0,12.0,70.0,0.0,0.0,unknown,rural,0.0,NaN,NaN,NaN
005815,0.0,0.0,NaN,NaN,0,0,0,0,0,0.0,0.0,2.0,10.0,55.0,unknown,2.0,10.0,55.0,0.0,0.0,unknown,rural,0.0,0.306250,0.065669,0.026775


In [12]:
# SPR20-597 HERS free-flow-speed method -> engineering baseline capacity (Ceng), Sections 2-3 of the doc.
# Uses each POI's POST-construction geometry, since Ceng is defined for post-construction geometry.
UNINTERRUPTED_CLASSES = {"freeway", "ramp"}
ONE_WAY_CLASSES = {"ramp"}  # lane counts on these are already one direction

# through_lanes in the project files is the two-way total. Set to False if yours are per direction.
THROUGH_LANES_ARE_TWO_WAY = True


def hers_ceng(functional_class, speed_limit, lane_width, signals_per_mile, stop_signs_per_mile, through_lanes):
    fc_norm = functional_class.map(normalize_functional_class)
    is_uninterrupted = fc_norm.isin(UNINTERRUPTED_CLASSES)
    sigma = np.where(is_uninterrupted, 0.1427, 0.3907)
    beta_hers = np.where(is_uninterrupted, 0.2092, 0.18378)

    VSPLIM = speed_limit.astype(float)
    base_ffs = (np.exp(sigma**2 / 2) * VSPLIM**(-1/beta_hers))**(-beta_hers)

    # Lane-width penalty (12ft=0, 11ft=1.5mph, 10ft=4mph, <10ft=6.5mph)
    lw = lane_width.astype(float)
    fw = np.select([lw >= 12, lw >= 11, lw >= 10], [0, 1.5, 4], default=6.5)
    ffs_lw = base_ffs - fw

    nsig = signals_per_mile.astype(float)
    zvdsig = np.where(nsig == 0, 0, 0.0687 * (1 - np.exp(-nsig / 24.4)))
    ffs_sig = 1 / (1/ffs_lw + zvdsig)

    nstp = stop_signs_per_mile.astype(float)
    zvdstp = np.where(nstp == 0, 0, nstp * (1.9 + 0.067 * ffs_sig))
    ffs_final = 1 / (1/ffs_sig + zvdstp / 1000)

    # Speed at capacity (HCM density = 35 pc/mi/ln) -> flow per lane
    v_cap = ffs_final * 0.9
    s_cap = v_cap * 35

    lanes = through_lanes.astype(float)
    if THROUGH_LANES_ARE_TWO_WAY:
        lanes = lanes.where(fc_norm.isin(ONE_WAY_CLASSES), lanes / 2)

    # Uninterrupted flow (no signals or stop signs): use HCM capacity by free-flow speed instead.
    # 35 pc/mi/ln is the LOS D/E density limit, not capacity (45), so the shortcut above undershoots
    # HCM by ~15-30% below 70 mph (e.g. a 50 mph freeway came out ~1,500 pc/h/ln vs HCM 2,200).
    no_control = (nsig.fillna(0) == 0) & (nstp.fillna(0) == 0)
    ramp_cap = np.select([ffs_lw > 50, ffs_lw > 40, ffs_lw > 30, ffs_lw > 20],
                         [2200, 2100, 2000, 1900], default=1800)            
    freeway_cap = np.clip(2200 + 10 * (ffs_lw - 50), 2000, 2400)             
    multilane_cap = np.clip(1900 + 20 * (ffs_lw - 45), 1700, 2300)           
    hcm_cap = np.select(
        [fc_norm.eq("ramp"), fc_norm.eq("freeway"), lanes >= 2],
        [ramp_cap, freeway_cap, multilane_cap],
        default=1700,                                                        
    )
    s_cap = np.where(no_control, hcm_cap, s_cap)

    return s_cap * lanes * f_hv


poi["engineering_capacity_ceng_pre"] = hers_ceng(
    poi["functional_class_pre"], poi["speed_limit_pre"], poi["lane_width_pre"],
    poi["signals_per_mile_pre"], poi["stop_signs_per_mile_pre"], poi["through_lanes_pre"],
)
poi["engineering_capacity_ceng_post"] = hers_ceng(
    poi["functional_class_post"], poi["speed_limit_post"], poi["lane_width_post"],
    poi["signals_per_mile_post"], poi["stop_signs_per_mile_post"], poi["through_lanes_post"],
)

poi[[
    "functional_class_post", "speed_limit_post", "through_lanes_pre", "through_lanes_post",
    "engineering_capacity_ceng_pre", "engineering_capacity_ceng_post",
]].head(10)

,functional_class_post,speed_limit_post,through_lanes_pre,through_lanes_post,engineering_capacity_ceng_pre,engineering_capacity_ceng_post
poi_id,,,,,,
001047,Principal Arterial,55.0,2.0,2.0,1615.0,1615.0
001A69,Principal Arterial,55.0,2.0,2.0,1615.0,1615.0
005017,unknown,55.0,2.0,2.0,1615.0,1615.0
005021,unknown,55.0,2.0,2.0,1615.0,1615.0
005602,unknown,70.0,6.0,6.0,6555.0,6555.0
005603,unknown,70.0,6.0,6.0,6555.0,6555.0
005604,unknown,70.0,6.0,6.0,6555.0,6555.0
005606,unknown,70.0,6.0,6.0,6555.0,6555.0
005815,unknown,55.0,2.0,2.0,1615.0,1615.0


In [13]:
CHANGE_TERMS = {
    "BLT": "d_left_turn_lanes",
    "BRT": "d_right_turn_lanes",
    "BE": "express_lane_added",
    "BS": "shoulder_widened",
    "BF": "functional_class_improved",
}

EXCLUDED_TERMS = {
    # already in Ceng (geometry)
    "BLM": "d_lane_miles",
    "BL": "d_through_lanes",
    "BW": "lane_width_improved",
    "BSL": "speed_limit_increased",
    # demand, not capacity
    "BA": "d_ln_aadt",
    "BP": "d_ln_pop",      # regional population growth, from us_county_socioeconomic_2000_2025.csv
    "BY": "d_ln_income",   # regional income growth, from us_county_socioeconomic_2000_2025.csv
}

_change_vars = poi[list(CHANGE_TERMS.values())].astype(float).replace([np.inf, -np.inf], np.nan)
_n_filled = _change_vars.isna().sum()
if _n_filled.any():
    print("Missing change terms treated as 0 (no change), POI counts:")
    print(_n_filled[_n_filled > 0].to_string())
_change_vars = _change_vars.fillna(0)

X_change = sum(betas[b] * _change_vars[col] for b, col in CHANGE_TERMS.items())
# BSIG (signals per mile) is left out: Ceng already applies the HERS signal-delay reduction,
# so the extra -10% per signal/mile counted signals twice.
X_level_pre = betas["BAT"] * poi["area_type_level"]
X_level_post = betas["BAT"] * poi["area_type_level"]

poi["capacity_multiplier_pre"] = np.exp(X_level_pre)
poi["capacity_multiplier"] = np.exp(X_change + X_level_post)
poi["peak_hour_capacity_pre"] = poi["engineering_capacity_ceng_pre"] * poi["capacity_multiplier_pre"]
poi["peak_hour_capacity"] = poi["engineering_capacity_ceng_post"] * poi["capacity_multiplier"]

# --- sanity check: flag capacities that are implausible per lane, and big lane-count jumps
_lanes_dir = np.where(
    THROUGH_LANES_ARE_TWO_WAY & ~poi["functional_class_post"].map(normalize_functional_class).isin(ONE_WAY_CLASSES),
    poi["through_lanes_post"] / 2, poi["through_lanes_post"],
)
poi["capacity_per_lane_post"] = poi["peak_hour_capacity"] / _lanes_dir
MAX_PLAUSIBLE_PER_LANE = 2400  # veh/hr/ln, roughly the HCM ceiling for a basic freeway segment
_suspect = poi[
    (poi["capacity_per_lane_post"] > MAX_PLAUSIBLE_PER_LANE)
    | ((poi["through_lanes_pre"] > 0) & (poi["through_lanes_post"] >= 3 * poi["through_lanes_pre"]))
]

# through_lanes = 0 (e.g. ramps with no lane count entered) would give a capacity of 0 -> leave blank instead
poi.loc[poi["through_lanes_pre"] == 0, "peak_hour_capacity_pre"] = np.nan
poi.loc[poi["through_lanes_post"] == 0, "peak_hour_capacity"] = np.nan

# Merge the POI-level results back onto every observation row, by POI AND period
_poi_out = pd.concat([
    poi[["engineering_capacity_ceng_pre", "capacity_multiplier_pre", "peak_hour_capacity_pre"]]
        .set_axis(["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"], axis=1)
        .assign(before_or_after="before"),
    poi[["engineering_capacity_ceng_post", "capacity_multiplier", "peak_hour_capacity"]]
        .set_axis(["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"], axis=1)
        .assign(before_or_after="after"),
]).reset_index()
master_df = master_df.drop(columns=["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"], errors="ignore")
master_df = master_df.merge(_poi_out, on=["poi_id", "before_or_after"], how="left")

Missing change terms treated as 0 (no change), POI counts:
d_left_turn_lanes     411
d_right_turn_lanes    414


## 8. Spot-check the merged data

Displays a narrowed view of just the columns most relevant for spot-checking, the join keys, dates, before/after labels, and a few geometric attributes (lane counts, shoulder width), to visually verify the merges worked correctly before writing anything out.

In [14]:
master_df[[
    "project_id", "poi_id", "observation_date", "date_open_actual",
    "before_or_after", "before_or_after_binary", "years_from_open",
    "through_lanes", "right_turn_lanes", "shoulder_width",
    "functional_class", "k_factor", "d_factor",
    "population", "avg annual pay (2025$)",
    "peak_hour_volume", "peak_hour_capacity",
]]

,project_id,poi_id,observation_date,date_open_actual,before_or_after,before_or_after_binary,years_from_open,through_lanes,right_turn_lanes,shoulder_width,functional_class,k_factor,d_factor,population,avg annual pay (2025$),peak_hour_volume,peak_hour_capacity
0,237506-2,701007,1997-01-01,2007-01-01,before,0,-10.0,4.0,0.0,6.0,principal arterial,0.12,0.60,NaN,NaN,2145.6,3446.425171
1,237506-2,701007,1998-01-01,2007-01-01,before,0,-9.0,4.0,0.0,6.0,principal arterial,0.12,0.60,NaN,NaN,2196.0,3446.425171
2,237506-2,701007,1999-01-01,2007-01-01,before,0,-8.0,4.0,0.0,6.0,principal arterial,0.12,0.60,NaN,NaN,2167.2,3446.425171
3,237506-2,701007,2000-01-01,2007-01-01,before,0,-7.0,4.0,0.0,6.0,principal arterial,0.12,0.60,477819.0,59976.37,2196.0,3446.425171
4,237506-2,701007,2001-01-01,2007-01-01,before,0,-6.0,4.0,0.0,6.0,principal arterial,0.12,0.60,486429.0,59622.17,2196.0,3446.425171
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7279,20190293,50076,2021-01-01,NaT,after,1,NaN,7.0,0.0,8.0,freeway,0.10,0.55,2328520.0,74991.32,3795.55,6465.575833
7280,20190293,50076,2022-01-01,NaT,after,1,NaN,7.0,0.0,8.0,freeway,0.10,0.55,2291363.0,71250.88,3930.41,6465.575833
7281,20190293,50076,2023-01-01,NaT,after,1,NaN,7.0,0.0,8.0,freeway,0.10,0.55,2314747.0,70773.11,5279.505,6465.575833
7282,20190293,50076,2024-01-01,NaT,after,1,NaN,7.0,0.0,8.0,freeway,0.10,0.55,2367034.0,69500.87,4966.005,6465.575833


## 9. Write out the combined CSV

Adds a blank `comment` column (for manual notes) as the very last column, then saves `master_df` to `combined_output.csv` inside `DATA_ROOT` (the same folder your project subfolders live in, alongside `validation_report.txt`) and confirms how many rows/columns were written. This file is the input for the quantile regression model-fitting notebook.

In [15]:
output_df = master_df.drop(
    columns=["engineering_capacity_ceng", "capacity_multiplier",
             "area_type_normalized", "project_type_category"],
    errors="ignore",
)

comment_cols = [c for c in output_df.columns if c.strip().lower() in ("comment", "comments")]
if comment_cols:
    comment_series = output_df[comment_cols[0]]
    for extra_col in comment_cols[1:]:
        comment_series = comment_series.fillna(output_df[extra_col])
    output_df = output_df.drop(columns=comment_cols)
else:
    comment_series = pd.Series(pd.NA, index=output_df.index)
output_df["comment"] = comment_series.fillna("")

output_path = os.path.join(DATA_ROOT, "combined_output.csv")
output_df.to_csv(output_path, index=False)
print(f"Wrote {len(output_df)} rows and {len(output_df.columns)} columns to {output_path}")

Wrote 7284 rows and 49 columns to C:\Users\User\Documents\Code\forecastcards_ba_data\data\combined_output.csv
